# AMADS Coding Notebooks

## Chord Bigrams and Equivalences

We build 24-by-24 grids of all major and minor triads and the pairs (bigrams) of chords therein.
Here we explore equivalence classes introduced in the eponymous `chord_bigrams` module.

See also the `chord_loops` notebook for an exploration of:
- What chord progressions are common (as measured in the Billboard corpus)?
- How can we measure similarity among (longer) progressions?

---

**By (author/s):** Mark Gotham

**For:** Attached to the
[AMADS code library](https://github.com/music-computing/amads/) and
["Keeping Score" book](https://doi.org/10.5281/zenodo.14938027),
but open to all.

**Licence:** MIT.

**Colour key:**
- <font color='green'> Green is for a block of information.
- <font color='purple'> Purple is for an exercise.
- <font color='crimson'> Crimson is for the solution to the exercise above it.

In [ ]:
from amads.core.pitch import Pitch, CHROMATIC_NAMES
from amads.core.chord import Chord
from amads.pitch.chord_bigram import ChordBigram

In [ ]:
# Preliminaries: build the ordered 24-chord axis: (C maj, C min, C# maj, C# min, ...)

QUALITIES = ('major', 'minor')

chords = [
    Chord(Pitch(name), q)
    for name in CHROMATIC_NAMES
    for q in QUALITIES
]

# Axis tick labels
tick_labels = [f"{c.root.name}{'M' if c.quality == 'major' else 'm'}" for c in chords]

print(f"{len(chords)}-length chord list prepared, starting {tick_labels[:6]} ... ")

This notebook will demonstrate various equivalence, always using:
- a 24-by-24 grid
- colour coding for some kind of equivalence

## <font color='purple'> Exercise: how many distinct options are there for each?

We will consider the following equivalence classes:
- 'Ø' stands for no equivalence reduction, so all pairs are distinct.
- 'I' is inversion equivalence: a major and minor triad at the same root are identified (they are inversions of one another, mod 12).
- 'R' is retrograde equivalence, so X-to-Y is equivalent to Y-to-X.
- 'K' is key (transposition) equivalence, so X-to-Y is equivalent to the same progression transposed by any amount.
- 'IR', 'IK', 'RK' are pairwise combinations of the above.
- 'IRK' combines all three, giving the coarsest classification.

Clearly 'Ø' has the most distinct options, and 'IRK' has the fewest. How many distinct classes are there for each?

Come up with your expected answer, with or without coding it.

## <font color='crimson'> Solution

The equivalence classes and their number of distinct progression classes are:

- **'Ø'** — no reduction; every ordered pair of chords is distinct. `24 × 24 = 576` (4 trivial).
- **'I'** — inversion equivalence halves the quality dimension (major ≡ minor at same root), reducing to 48 classes (2 trivial).
- **'R'** — retrograde equivalence; X→Y ≡ Y→X. Almost halves the grid, accounting for the 24 diagonal identity pairs: 48 classes (2 trivial).
- **'K'** — transposition equivalence; removes absolute pitch-class information. `576 / 12 = 48` classes (2 trivial).
- **'IR'** — I + R; 24 classes (1 trivial).
- **'IK'** — I + K; 24 classes (1 trivial).
- **'RK'** — R + K; three unordered quality pairs (MM, Mm, mm) × 7 interval classes (0–6) = 26 classes (the spec counts 26, including 2 trivial).
- **'IRK'** — all three combined; the coarsest classification. 13 classes (1 trivial).

Note: *trivial* classes are those where chord1 = chord2 (identity progressions).

Now let's code this simply by exhaustively stepping over all pairs in a given class.

We'll also leave a print statement in to verify the numbers above and make this a shared function for grid building and colour allocation that can be used later on.


In [ ]:
import numpy as np

# Equivalences that require a key pitch class for the label computation
_NEEDS_KEY = {'Ø', 'I', 'R', 'IR'}

def build_grid(equivalence, key_pc=None) -> tuple[list[list[ChordBigram]], np.array, dict]:
    """
    Build a 24x24 matrix with each cell as a `ChordBigram` object.
    Also build a parallel integer matrix mapping each cell to a colour index (distinct class ID).
    This is how we will navigate and visualise equivalence classes.

    `key_pc` is required for equivalences in {'Ø', 'I', 'R', 'IR'} and ignored otherwise.

    Returns: bigrams (list of lists), class_matrix (np array), class_to_label (dict)
    """
    bigrams = []
    label_to_id = {}
    id_matrix = np.zeros((24, 24), dtype=int)

    kpc = key_pc if equivalence in _NEEDS_KEY else None

    for i, c1 in enumerate(chords):
        row = []
        for j, c2 in enumerate(chords):
            b = ChordBigram(c1, c2, equivalence, kpc)
            row.append(b)
            lbl = b.label
            if lbl not in label_to_id:
                label_to_id[lbl] = len(label_to_id)
            id_matrix[i, j] = label_to_id[lbl]
        bigrams.append(row)

    id_to_label = {v: k for k, v in label_to_id.items()}
    print(f"Equivalence {equivalence!r}: {len(label_to_id)} distinct classes")
    return bigrams, id_matrix, id_to_label

In [ ]:
KEY_PC = 0  # Used for Ø, I, R, IR grids

grid_exact, mat_exact, cls_exact = build_grid('Ø',  KEY_PC)
grid_I,     mat_I,     cls_I     = build_grid('I',  KEY_PC)
grid_R,     mat_R,     cls_R     = build_grid('R',  KEY_PC)
grid_K,     mat_K,     cls_K     = build_grid('K')
grid_IR,    mat_IR,    cls_IR    = build_grid('IR', KEY_PC)
grid_IK,    mat_IK,    cls_IK    = build_grid('IK')
grid_RK,    mat_RK,    cls_RK    = build_grid('RK')
grid_IRK,   mat_IRK,   cls_IRK   = build_grid('IRK')

---

## <font color='green'> Plotting

Now for the potting, we'll do distinct plots for each equivalence class based on what's most useful and readily visualised.

In [ ]:
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# Suffix pattern: equivalence tag at end of label (e.g. 'Ø', 'IR', 'IRK', 'RK', 'IK', 'I', 'R', 'K')
_EQUIV_SUFFIX_RE = re.compile(r'(Ø|IRK|IR|IK|RK|I|R|K)$')

def make_cmap(n, base_cmap='tab20'):
    """Build a ListedColormap with exactly n colours."""
    if n <= 20:
        base = plt.get_cmap('tab20')
    elif n <= 40:
        # interleave tab20b and tab20c
        c1 = plt.get_cmap('tab20b').colors
        c2 = plt.get_cmap('tab20c').colors
        colors = [c for pair in zip(c1, c2) for c in pair]
        return ListedColormap(colors[:n])
    else:
        base = plt.get_cmap('gist_ncar')
        colors = [base(i / n) for i in range(n)]
        return ListedColormap(colors)
    colors = [base(i) for i in range(min(n, base.N))]
    return ListedColormap(colors[:n])

def plot_grid(ax, id_matrix, id_to_label, title, show_labels=True, max_legend=48):
    n_classes = id_matrix.max() + 1
    cmap = make_cmap(n_classes)

    ax.imshow(id_matrix, cmap=cmap, vmin=0, vmax=n_classes - 1,
              interpolation='nearest', aspect='equal')

    # Cell text — only if few enough classes to be legible
    if show_labels and n_classes <= 48:
        for i in range(24):
            for j in range(24):
                lbl = id_to_label[id_matrix[i, j]]
                # Strip equivalence suffix for brevity inside cell
                short = _EQUIV_SUFFIX_RE.sub('', lbl).strip()
                ax.text(j, i, short, ha='center', va='center',
                        fontsize=4.5, color='black', clip_on=True)

    # Axes ticks
    ax.set_xticks(range(24))
    ax.set_yticks(range(24))
    ax.set_xticklabels(tick_labels, rotation=90, fontsize=6)
    ax.set_yticklabels(tick_labels, fontsize=6)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=6)

    # Grid lines between cells
    ax.set_xticks(np.arange(-0.5, 24, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 24, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=0.3)
    ax.tick_params(which='minor', length=0)

    # Legend (class id -> label), capped
    if n_classes <= max_legend:
        patches = [
            mpatches.Patch(color=cmap(i / (n_classes - 1) if n_classes > 1 else 0),
                           label=id_to_label[i])
            for i in range(n_classes)
        ]
        ax.legend(handles=patches, bbox_to_anchor=(1.02, 1), loc='upper left',
                  fontsize=5, frameon=True, ncol=1 if n_classes <= 24 else 2)

    return n_classes

---

## Ø — Exact (no equivalence reduction)

As all 576 bigrams are distinct, there are too many to have one colour per class and too many to include labels.

Here, we'll simply show 4 colours for each pair of chord qualities:
- major-major (MM),
- major-minor (Mm),
- minor-major (mM),
- minor-minor (mm)

In [ ]:
quality_pair_matrix = np.array([
    [2 * (chords[i].quality == 'major') + (chords[j].quality == 'major')
     for j in range(24)]
    for i in range(24)
])
qp_labels = {0: 'mm', 1: 'mM', 2: 'Mm', 3: 'MM'}
qp_cmap = ListedColormap(['#4e79a7','#f28e2b','#e15759','#76b7b2'])

fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(quality_pair_matrix, cmap=qp_cmap, vmin=0, vmax=3,
          interpolation='nearest', aspect='equal')
ax.set_xticks(range(24))
ax.set_yticks(range(24))
ax.set_xticklabels(tick_labels, rotation=90, fontsize=6)
ax.set_yticklabels(tick_labels, fontsize=6)
ax.set_title('Ø — Exact (576 distinct classes)\nColour = quality pair only', fontsize=11, fontweight='bold')
ax.set_xticks(np.arange(-0.5, 24, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 24, 1), minor=True)
ax.grid(which='minor', color='white', linewidth=0.3)
ax.tick_params(which='minor', length=0)
patches = [mpatches.Patch(color=qp_cmap(i/3), label=qp_labels[i]) for i in range(4)]
ax.legend(handles=patches, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.grid(False)  # Override defaults as needed
plt.tight_layout()
plt.savefig('grid_exact.pdf', bbox_inches='tight')
plt.show()

---

## R — Retrograde invariant

With Retrograde equivalence (R), the upper and lower triangles are mirror images of one another.

Here we'll plot twice:
1. Colour quality-pairs broadly as above but noting the Mm is unordered, so we now plot only 3 classes.
2. Show the almost-half symmetry, highlighting the two equivalent regions and the 24 exceptions cases along the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: quality-pair colouring showing symmetry
ax = axes[0]
# For R the quality pair is unordered: {MM, Mm/mM, mm} -> 3 classes
r_qp_matrix = np.array([
    [min(2*(chords[i].quality=='major')+(chords[j].quality=='major'),
         2*(chords[j].quality=='major')+(chords[i].quality=='major'))
     for j in range(24)]
    for i in range(24)
])
r_qp_labels = {0: 'mm', 1: 'Mm (unordered)', 2: 'MM'}
r_cmap = ListedColormap(['#4e79a7','#f28e2b','#76b7b2'])
ax.imshow(r_qp_matrix, cmap=r_cmap, vmin=0, vmax=2, interpolation='nearest', aspect='equal')
ax.set_xticks(range(24)); ax.set_yticks(range(24))
ax.set_xticklabels(tick_labels, rotation=90, fontsize=6)
ax.set_yticklabels(tick_labels, fontsize=6)
ax.set_title('R — quality-pair structure\n(symmetric across diagonal)', fontsize=10, fontweight='bold')
ax.set_xticks(np.arange(-0.5,24,1), minor=True); ax.set_yticks(np.arange(-0.5,24,1), minor=True)
ax.grid(False)
ax.plot([-0.5,23.5],[-0.5,23.5], color='white', linewidth=1, linestyle='--', alpha=0.6)
patches = [mpatches.Patch(color=r_cmap(i/2), label=r_qp_labels[i]) for i in range(3)]
ax.legend(handles=patches, bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)

# Right: highlight upper vs lower triangle
ax2 = axes[1]
triangle_matrix = np.zeros((24,24))
for i in range(24):
    for j in range(24):
        if i < j:  triangle_matrix[i,j] = 1  # upper
        elif i > j: triangle_matrix[i,j] = 2  # lower
        # diagonal = 0
ax2.imshow(triangle_matrix, cmap=ListedColormap(['#aec7e8','#ffbb78','#98df8a']),
           vmin=0, vmax=2, interpolation='nearest', aspect='equal')
ax2.set_xticks(range(24)); ax2.set_yticks(range(24))
ax2.set_xticklabels(tick_labels, rotation=90, fontsize=6)
ax2.set_yticklabels(tick_labels, fontsize=6)
ax2.set_title('R — triangle equivalence\n(upper ↔ lower = same class)', fontsize=10, fontweight='bold')
ax2.set_xticks(np.arange(-0.5,24,1), minor=True); ax2.set_yticks(np.arange(-0.5,24,1), minor=True)
ax2.grid(False)
ax2.plot([-0.5,23.5],[-0.5,23.5], color='white', linewidth=1.5, linestyle='--')
patches2 = [mpatches.Patch(color=c, label=l) for c,l in
            zip(['#aec7e8','#ffbb78','#98df8a'], ['diagonal (self-pair)','upper triangle','lower triangle'])]
ax2.legend(handles=patches2, bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('grid_R.pdf', bbox_inches='tight')
plt.show()

---

## K — Key/transposition invariant

With only 48 distinct classes and with K removing the need for supscripts in labels,
we can fully colour-code all 48 complete with full labels (excepting the K which obviously applies to all).

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
plot_grid(ax, mat_K, cls_K,
          title=f'K — Transposition invariant ({mat_K.max()+1} distinct classes)',
          show_labels=True, max_legend=48)
ax.grid(False)
plt.tight_layout()
plt.savefig('grid_K.pdf', bbox_inches='tight')
plt.show()

---

## RK — Retrograde + Key invariant

With R + K equivalence there are 26 distinct classes: three unordered quality pairs (MM, Mm, mm)
× 7 interval classes (ic 0–6), minus 1 because both trivial ic-0 classes (MM and mm) collapse
into the identity, giving 26 total (2 trivial).

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
plot_grid(ax, mat_RK, cls_RK,
          title=f'RK — Retrograde + Transposition invariant ({mat_RK.max()+1} distinct classes)',
          show_labels=True, max_legend=48)
ax.grid(False)
plt.tight_layout()
plt.savefig('grid_RK.pdf', bbox_inches='tight')
plt.show()

---

## I — Inversion invariant

Under I equivalence, a major and minor triad at the same root are identified (they are
mod-12 inversions of one another). This halves the effective quality alphabet, reducing
576 classes to 48 (2 trivial).

The label always starts with **M** (since major/minor are merged); the last letter is
**M** if both triads share the same quality, **m** if they differ. The middle interval
is the opci from the tonicized root to the non-tonicized root — if the tonicized triad
is minor, this interval is subtracted from 12 (equivalently, the direction is reversed).

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
plot_grid(ax, mat_I, cls_I,
          title=f'I — Inversion invariant ({mat_I.max()+1} distinct classes)',
          show_labels=True, max_legend=48)
ax.grid(False)
plt.tight_layout()
plt.savefig('grid_I.pdf', bbox_inches='tight')
plt.show()

---

## IR — Inversion + Retrograde invariant

Combining I and R equivalence yields 24 classes (1 trivial). The near/far sort from R
determines the tonicized triad, and the I-normalisation then adjusts the interval if
that triad is minor.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
plot_grid(ax, mat_IR, cls_IR,
          title=f'IR — Inversion + Retrograde invariant ({mat_IR.max()+1} distinct classes)',
          show_labels=True, max_legend=48)
ax.grid(False)
plt.tight_layout()
plt.savefig('grid_IR.pdf', bbox_inches='tight')
plt.show()

---

## IK — Inversion + Key invariant

Combining I and K equivalence also gives 24 classes (1 trivial). Transposition removes
absolute pitch information; inversion merges the quality pairs. The interval is computed
chronologically from chord1 to chord2, adjusted for inversion if chord1 is minor.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
plot_grid(ax, mat_IK, cls_IK,
          title=f'IK — Inversion + Key invariant ({mat_IK.max()+1} distinct classes)',
          show_labels=True, max_legend=48)
ax.grid(False)
plt.tight_layout()
plt.savefig('grid_IK.pdf', bbox_inches='tight')
plt.show()

---

## IRK — Inversion + Retrograde + Key invariant

The coarsest classification, combining all three equivalences, yields 13 classes (1 trivial).
Each class is characterised only by the interval class (ic) between the two roots and whether
the triads share the same quality or not.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
plot_grid(ax, mat_IRK, cls_IRK,
          title=f'IRK — Inversion + Retrograde + Key invariant ({mat_IRK.max()+1} distinct classes)',
          show_labels=True, max_legend=48)
ax.grid(False)
plt.tight_layout()
plt.savefig('grid_IRK.pdf', bbox_inches='tight')
plt.show()

End

---